# Titanic Survival Prediction with scikit-learn

**Goal:** predict whether a passenger survived the Titanic using simple features (class, sex, age, family size, fare, port).

**Dataset:** Titanic passenger data (loaded through seaborn, so no Kaggle login is needed).

**Models compared:** Logistic Regression, Decision Tree, Random Forest.

**Author:** Fathir

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

sns.set_theme(style="whitegrid")

## 2. Load the data

In [ ]:
df = sns.load_dataset("titanic")
print("Rows, columns:", df.shape)
df.head()

## 3. Explore the data

Before building any model, look at the data: what columns exist, what is missing, and what patterns are visible.

In [ ]:
df.info()

In [ ]:
# Which columns have missing values?
df.isna().sum().sort_values(ascending=False).head(5)

In [ ]:
print(f"Overall survival rate: {df['survived'].mean():.1%}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.barplot(data=df, x="sex", y="survived", ax=axes[0])
axes[0].set_title("Survival rate by sex")

sns.barplot(data=df, x="pclass", y="survived", ax=axes[1])
axes[1].set_title("Survival rate by passenger class")

sns.histplot(data=df, x="age", hue="survived", bins=30, ax=axes[2])
axes[2].set_title("Age distribution by survival")

plt.tight_layout()
plt.show()

**What I notice:** *(write 2-3 sentences in your own words after you run the charts above, for example which group survived the most and which the least.)*

## 4. Prepare the features

Steps:
1. Keep only the columns we need.
2. Fill the 2 missing `embarked` values with the most common port ("S").
3. Turn text columns into numbers (`sex` becomes 0/1, `embarked` becomes one-hot columns).
4. Create a new feature, `family_size`, from `sibsp` + `parch` + 1.

In [ ]:
data = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()

data["embarked"] = data["embarked"].fillna("S")
data["sex"] = data["sex"].map({"male": 0, "female": 1})
data["family_size"] = data["sibsp"] + data["parch"] + 1
data = pd.get_dummies(data, columns=["embarked"], drop_first=True)

data.head()

## 5. Split into train and test sets

The model learns from the training set, and the test set (data it has never seen) tells us how well it really works.

`age` still has missing values, so we fill them with the **median age of the training set only**. Using the training median for both sets avoids leaking information from the test set.

In [ ]:
X = data.drop(columns="survived")
y = data["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

age_median = X_train["age"].median()
X_train["age"] = X_train["age"].fillna(age_median)
X_test["age"] = X_test["age"].fillna(age_median)

print("Train size:", X_train.shape, "| Test size:", X_test.shape)

## 6. Train and compare three models

- **Logistic Regression:** a simple, fast baseline (it needs scaled features, so we add a `StandardScaler`).
- **Decision Tree:** learns yes/no rules; we limit `max_depth` so it does not memorise the training data.
- **Random Forest:** many decision trees voting together, usually more accurate than one tree.

We report test accuracy, F1 score, and 5-fold cross-validation accuracy (a more stable estimate than one split).

In [ ]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
    rows.append({
        "Model": name,
        "Test accuracy": accuracy_score(y_test, preds),
        "Test F1": f1_score(y_test, preds),
        "CV accuracy (mean)": cv.mean(),
    })

results = pd.DataFrame(rows).set_index("Model").round(3)
results

In [ ]:
results["Test accuracy"].plot(kind="bar", figsize=(6, 4), rot=0, color=["#4285F4", "#EA4335", "#34A853"])
plt.ylim(0.6, 1.0)
plt.ylabel("Test accuracy")
plt.title("Model comparison")
plt.tight_layout()
plt.show()

## 7. Look closer at the best model

A confusion matrix shows where the model is right and wrong, and feature importance shows which inputs mattered most.

In [ ]:
best_name = results["Test accuracy"].idxmax()
best_model = models[best_name]
best_preds = best_model.predict(X_test)
print("Best model on the test set:", best_name)

ConfusionMatrixDisplay(confusion_matrix(y_test, best_preds), display_labels=["Died", "Survived"]).plot(cmap="Blues")
plt.title(f"Confusion matrix: {best_name}")
plt.show()

print(classification_report(y_test, best_preds, target_names=["Died", "Survived"]))

In [ ]:
# Feature importance from the Random Forest
rf = models["Random Forest"]
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values()

importance.plot(kind="barh", figsize=(6, 4), color="#4285F4")
plt.title("Random Forest feature importance")
plt.tight_layout()
plt.show()

## 8. Conclusion

*(Write this in your own words after you see your results. Suggested points:)*

- Which model scored best, and by how much?
- Which features mattered most, and does that match what you saw in the charts?
- What are the limits of this project? (small dataset, no hyperparameter tuning, a single train/test split)

## 9. Next steps

- Tune the models with `GridSearchCV`.
- Try more features, such as titles extracted from names.
- Move on to the next project: image classification.